<a href="https://colab.research.google.com/github/BenFreilichUVA/ds2002-fa26/blob/main/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [13]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [14]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [15]:
# TODO
vendors = pd.json_normalize(vendors_json)
joined = pd.merge(lines, vendors, on='vendor_id', how='left', indicator=True)

unmatched = joined[joined['_merge'] == 'left_only']
print('The unmatched vendor(s) is:', unmatched['vendor_id'].unique().tolist())
print('The total quantity of the unmatched vendor(s) is:', unmatched['qty'].sum())
joined

The unmatched vendor(s) is: ['V-07']
The total quantity of the unmatched vendor(s) is: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [16]:
# TODO
names = joined.assign(name=joined['name'].fillna('unmatched'))
VendorSorted = names.groupby('name')['qty'].sum().sort_values(ascending=False)
assert VendorSorted.sum() == 7
VendorSorted
# The table now simply displays the name of the unmatched vendor as unmatched. This is because the code to create the "names" table fills any vendor name that is NaN with unmatched.


,qty
name,
Hoos Burgers,3
unmatched,3
Cav Merch,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [17]:
ZoneSorted = joined.groupby('zone')['qty'].sum()
print(ZoneSorted)
print('Total per zone:', ZoneSorted.sum(), '| Amount missing from total of seven:', lines['qty'].sum() - ZoneSorted.sum())
print('Zone A sold the most, but since it is the only zone in the table with sales, this makes sense.')
print('The missing units are from the hot dog vendor, which does not have a zone assigned.')

zone
A    4
Name: qty, dtype: int64
Total per zone: 4 | Amount missing from total of seven: 3
Zone A sold the most, but since it is the only zone in the table with sales, this makes sense.
The missing units are from the hot dog vendor, which does not have a zone assigned.


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [18]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

rows = []
for i in ragged_json:
    if 'lines' in i:
        for j in i['lines']:
            rows.append([i['order'], i['vendor_id'], j.get('item'), j.get('qty')])
RaggedFrame = pd.DataFrame(rows, columns=['order', 'vendor_id', 'item', 'qty'])
RaggedFrame

,order,vendor_id,item,qty
0,4,V-01,Soda,2.0
1,6,V-01,Fries,NaN


### Q6 — Validate

In [19]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?


**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?



A) One row per line is helpful because each question in the lab is a sum of quantity. We want to make sure we are looking at how the total quantities are factoring by vendor, zone, etc., which is easiest when you display it line by line. Total units per vendor would be hard to calculate taking the table as is, if we instead break it up into rows, it is much easier to look at and compute what the quantities are for each individual characteristic.

B) A quantity of zero means we know that none were sold. A missing value means we don't know what was sold. It is possible that nothing was sold there, but we cannot make any guarantee of that. If we get rid of all values where we don't know a certain value, we will lose a lot of important data. Downstream, the totals will end up being understated without warning. Because we are dropping values with NaN, we are losing a total amount off the total that we had at the start. A good example appears in Q5. If we just drop order 6 because we do not know the quantity, we are losing the actual total. It may not be very helpful to us to just know that the value is missing, but it is better than just filling it with a zero and moving on. It would also likely confuse an analyst, as someone ordering something and buying zero units would be very confusing.